# CUDA Graphs vs `torch.compile` for Qwen3 Embedding Models

### The question

`torch.compile()` speeds up a Qwen3 embedding model. Would **manual CUDA Graph capture** help *beyond* that?

### Key context

- `torch.compile(mode="reduce-overhead")` **already** wraps the compiled region in a CUDA Graph internally.
- So the real question is: does capturing a *wider* scope (the entire forward pass) via manual `cuda.CUDAGraph()` beat what the compiler does on its own?
- Embedding models are especially CUDA-Graph-friendly because input shapes are **static** at serving time (fixed `max_seq_len` + padding).

### What we benchmark

| Method | Operator fusion | CUDA Graph | Who captures the graph? |
|---|---|---|---|
| **eager** | no | no | — |
| **compile** | yes | no | — |
| **compile(reduce-overhead)** | yes | yes | the compiler (narrow scope) |
| **manual graph** | no | yes | us (wide scope: entire forward) |
| **compile + manual graph** | yes | yes | us (wide scope, on compiled model) |

## 1. Setup — installs, imports, GPU info

In [ ]:
# !pip install transformers accelerate --quiet

In [1]:
import gc, time, os
from contextlib import contextmanager
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

print(f"PyTorch    {torch.__version__}")
print(f"CUDA avail  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU         {props.name}  ({props.total_memory / 1024**3:.1f} GiB)")
    print(f"Compute cap {props.major}.{props.minor}")

PyTorch    2.12.0+cu130
CUDA avail  True
GPU         NVIDIA GeForce RTX 5070 Ti  (15.5 GiB)
Compute cap 12.0


## 2. Timer helper

In [2]:
# Global results dict populated by benchmark() as each method runs
results = {}

@contextmanager
def gpu_timer(name: str = ""):
    """Wall-clock + GPU-synchronised timer via CUDA events."""
    start_ev = torch.cuda.Event(enable_timing=True)
    end_ev   = torch.cuda.Event(enable_timing=True)
    t0 = time.perf_counter()
    start_ev.record()
    yield
    end_ev.record()
    torch.cuda.synchronize()
    wall_ms = (time.perf_counter() - t0) * 1000
    gpu_ms  = start_ev.elapsed_time(end_ev)
    return (name, wall_ms, gpu_ms)


def benchmark(fn, iters: int = 100, warmup_iters: int = 10, label: str = ""):
    """Warm up, then record `iters` synchronised timing samples.
    Stores result in the global `results` dict and returns (avg_ms, p50_ms, p99_ms)."""
    # warmup
    for _ in range(warmup_iters):
        fn()
    torch.cuda.synchronize()

    gc.collect()
    torch.cuda.empty_cache()

    samples = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        samples.append((time.perf_counter() - t0) * 1000)

    avg = sum(samples) / len(samples)
    srt = sorted(samples)
    p50 = srt[len(srt) // 2]
    p99 = srt[int(len(srt) * 0.99)]
    print(f"  {label:<24s} avg={avg:7.3f}ms  p50={p50:7.3f}ms  p99={p99:7.3f}ms")
    results[label] = {"avg": avg, "p50": p50, "p99": p99}
    return avg, p50, p99

## 3. Download the Qwen3-Embedding-0.6B model

In [3]:
MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
DTYPE    = torch.bfloat16  # bfloat16 saves memory, good on A100/A10/L4+

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model (this downloads ~1.2 GB on first run) ...")
model = AutoModel.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    trust_remote_code=True,
).cuda()
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {n_params / 1e6:.1f}M")
print(f"  Model dtype: {next(model.parameters()).dtype}")
print(f"  Max position embeddings: {getattr(model.config, 'max_position_embeddings', '?')}")

Loading tokenizer ...
Loading model (this downloads ~1.2 GB on first run) ...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Parameters: 595.8M
  Model dtype: torch.bfloat16
  Max position embeddings: 32768


### Quick sanity check — does it produce sensible embeddings?

In [4]:
texts = ["Hello world", "The quick brown fox jumps over the lazy dog"]
encoded = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
input_ids, attention_mask = encoded["input_ids"].cuda(), encoded["attention_mask"].cuda()

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    hidden = outputs.last_hidden_state  # (B, L, D)

# Mean pooling over non-padded tokens (standard for sentence embeddings)
mask_expanded = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
embeddings = (hidden * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1e-9)

print(f"Hidden states shape: {hidden.shape}")
print(f"Embedding shape:    {embeddings.shape}")

# Cosine similarity between the two sentences
cos = F.cosine_similarity(embeddings[0:1], embeddings[1:2])
print(f"Cosine similarity:  {cos.item():.4f}")

Hidden states shape: torch.Size([2, 10, 1024])
Embedding shape:    torch.Size([2, 1024])
Cosine similarity:  0.6331


## 4. Build dummy input (fixed-shape, like a serving workload)

In [5]:
# We use a fixed sequence length — this is the key requirement for CUDA Graphs.
BATCH_SIZE = 4
MAX_SEQ_LEN = 512

dummy_texts = [
    "The CUDA Graphs feature in PyTorch allows capturing a sequence of GPU operations "
    "as a static graph that can be replayed with a single kernel launch, significantly "
    "reducing CPU overhead for workloads with fixed input shapes such as embedding models "
    "deployed behind a serving endpoint with a fixed maximum sequence length."] * BATCH_SIZE
enc = tokenizer(
    dummy_texts,
    padding="max_length",
    truncation=True,
    max_length=MAX_SEQ_LEN,
    return_tensors="pt",
)
input_ids     = enc["input_ids"].cuda()
attention_mask = enc["attention_mask"].cuda()

print(f"input_ids shape:      {input_ids.shape}")
print(f"attention_mask shape: {attention_mask.shape}")
print(f"dtype: {input_ids.dtype}")

input_ids shape:      torch.Size([4, 512])
attention_mask shape: torch.Size([4, 512])
dtype: torch.int64


## 5. Warmup + benchmark — each method in one block

Each block below warms up (which also triggers compilation / graph capture for the compiled methods), then immediately runs the timed benchmark.  Results accumulate into the global `results` dict.

In [6]:
ITERS       = 200
WARMUP      = 20

# Helper: full forward → mean pooled embedding
def embed(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean pooling over non-padded tokens."""
    m = mask.unsqueeze(-1).expand(hidden.size()).float()
    return (hidden * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-9)


def eager_forward(ids, am):
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=am).last_hidden_state
    return embed(out, am)

In [7]:
# ══════════════════════════════════════════════════════════════════════
# 5.1  Eager (baseline)
# ══════════════════════════════════════════════════════════════════════
print("─" * 50)
benchmark(lambda: eager_forward(input_ids, attention_mask),
          iters=ITERS, warmup_iters=WARMUP, label="eager")

──────────────────────────────────────────────────
  eager                    avg= 43.873ms  p50= 43.869ms  p99= 44.036ms


(43.87330891007878, 43.86876900025527, 44.035619001078885)

In [8]:
# ══════════════════════════════════════════════════════════════════════
# 5.2  torch.compile (default — no CUDA Graph)
# ══════════════════════════════════════════════════════════════════════
@torch.compile
def compiled_forward(ids, am):
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=am).last_hidden_state
    return embed(out, am)

benchmark(lambda: compiled_forward(input_ids, attention_mask),
          iters=ITERS, warmup_iters=WARMUP, label="compile")

nvcc warning : incompatible redefinition for option 'compiler-bindir', the last value of this option was used


  compile                  avg= 31.753ms  p50= 31.746ms  p99= 31.978ms


(31.75332864495431, 31.74597199904383, 31.978245999198407)

In [9]:
# ══════════════════════════════════════════════════════════════════════
# 5.3  torch.compile(reduce-overhead) — CUDA Graph inside the compiler
# ══════════════════════════════════════════════════════════════════════
@torch.compile(mode="reduce-overhead")
def compiled_ro_forward(ids, am):
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=am).last_hidden_state
    return embed(out, am)

benchmark(lambda: compiled_ro_forward(input_ids, attention_mask),
          iters=ITERS, warmup_iters=WARMUP, label="compile(reduce-overhead)")

  compile(reduce-overhead) avg= 31.276ms  p50= 31.243ms  p99= 31.475ms


(31.276345659971412, 31.242936998751247, 31.47528699992108)

In [10]:
# ══════════════════════════════════════════════════════════════════════
# 5.4  Manual CUDA Graph (eager model, wide scope)
# ══════════════════════════════════════════════════════════════════════
static_ids = input_ids.clone()
static_am  = attention_mask.clone()
_ = eager_forward(static_ids, static_am)  # steady allocator before capture

manual_graph = torch.cuda.CUDAGraph()
with torch.cuda.graph(manual_graph):
    with torch.no_grad():
        _mg_out = model(input_ids=static_ids, attention_mask=static_am).last_hidden_state
    _mg_emb = embed(_mg_out, static_am)

def manual_graph_forward(ids, am):
    static_ids.copy_(ids)
    static_am.copy_(am)
    manual_graph.replay()
    return _mg_emb

benchmark(lambda: manual_graph_forward(input_ids, attention_mask),
          iters=ITERS, warmup_iters=WARMUP, label="manual CUDA Graph")

  manual CUDA Graph        avg= 41.199ms  p50= 41.213ms  p99= 41.265ms


(41.19870807007828, 41.2131569974008, 41.265007999754744)

In [11]:
# ══════════════════════════════════════════════════════════════════════
# 5.5  Hybrid: torch.compile + manual CUDA Graph
# ══════════════════════════════════════════════════════════════════════
compiled_model = torch.compile(model, mode="default")
static_ids2 = input_ids.clone()
static_am2  = attention_mask.clone()

with torch.no_grad():
    _ = compiled_model(input_ids=static_ids2, attention_mask=static_am2)

compile_graph = torch.cuda.CUDAGraph()
with torch.cuda.graph(compile_graph):
    with torch.no_grad():
        _cg_out = compiled_model(input_ids=static_ids2, attention_mask=static_am2).last_hidden_state
    _cg_emb = embed(_cg_out, static_am2)

def compile_graph_forward(ids, am):
    static_ids2.copy_(ids)
    static_am2.copy_(am)
    compile_graph.replay()
    return _cg_emb

benchmark(lambda: compile_graph_forward(input_ids, attention_mask),
          iters=ITERS, warmup_iters=WARMUP, label="compile + CUDA Graph")

  compile + CUDA Graph     avg= 31.115ms  p50= 31.144ms  p99= 31.162ms


(31.11542943503082, 31.144388998654904, 31.16168800261221)

## 6. Results summary

Each method already printed its timing above.  The global `results` dict holds all the numbers for the table below.

In [ ]:
# (benchmark loop now lives inside each method's block in section 5)

## 7. Analyse the results

In [12]:
eager_avg = results["eager"]["avg"]
print(f"Eager baseline (avg): {eager_avg:.3f} ms\n")

print(f"{'Method':<30s} {'avg ms':>9s}  {'p50 ms':>9s}  {'p99 ms':>9s}  {'speedup':>8s}")
print("-" * 75)
for label, r in results.items():
    su = eager_avg / r["avg"]
    best = max(eager_avg / v["avg"] for v in results.values())
    marker = "  ← best!" if su == best else ""
    print(f"{label:<30s} {r['avg']:9.3f}  {r['p50']:9.3f}  {r['p99']:9.3f}  {su:6.2f}x{marker}")

Eager baseline (avg): 43.873 ms

Method                            avg ms     p50 ms     p99 ms   speedup
---------------------------------------------------------------------------
eager                             43.873     43.869     44.036    1.00x
compile                           31.753     31.746     31.978    1.38x
compile(reduce-overhead)          31.276     31.243     31.475    1.40x
manual CUDA Graph                 41.199     41.213     41.265    1.06x
compile + CUDA Graph              31.115     31.144     31.162    1.41x  ← best!


### Observations

After running, look for these patterns:

| Pattern | What it means |
|---|---|
| `compile(ro)` ≈ `compile + graph` | The compiler's internal CUDA Graph is already capturing the right scope. Manual graph adds nothing. |
| `compile + graph` > `compile(ro)` | Manual capture outside the compiler grabs ops the compiler couldn't fuse — you beat the compiler. |
| `manual graph` close to `compile(ro)` | Even without operator fusion, launch overhead dominates. The model is CPU-bound per-invocation. |
| `compile` ≈ `compile(ro)` | GPU compute dominates; launch overhead is negligible → CUDA Graphs don't help much here. |

### Why these results?

For a **0.6B embedding model** at batch=1:
- The GPU is underutilised — many small kernel launches.
- CPU launch overhead *can* be a meaningful fraction of total latency.
- CUDA Graphs eliminate that overhead regardless of who captures them.
- **The winner is whoever captures the widest scope without breaking.**

For a **7B model** at larger batch, compute dominates and the picture changes — all methods converge.

## 8. (Optional) Probe: try different sequence lengths

The longer the sequence, the more GPU compute per forward pass, the less launch overhead matters. CUDA Graph gains shrink as `seq_len` grows.

In [ ]:
for sl in [64, 128, 256, 512, 1024]:
    # Build new inputs
    t = ["x"] * BATCH_SIZE
    e = tokenizer(t, padding="max_length", truncation=True, max_length=sl, return_tensors="pt")
    ids = e["input_ids"].cuda()
    am  = e["attention_mask"].cuda()

    # Quick eager run
    t0 = time.perf_counter()
    for _ in range(50):
        eager_forward(ids, am)
    torch.cuda.synchronize()
    eager_ms = (time.perf_counter() - t0) / 50 * 1000

    print(f"seq_len={sl:5d}  eager avg={eager_ms:.3f}ms")

## 9. (Optional) Try the 7B model

Swap the model ID and dtype below, then re-run cells 3–7.

```python
MODEL_ID = "Qwen/Qwen3-Embedding-7B"
DTYPE    = torch.bfloat16   # 7B in fp16 ~14 GB; bfloat16 fits on 24 GB cards
```

---

## Bottom line

**`torch.compile(mode="reduce-overhead")` is the right default.** It gives you operator fusion *and* CUDA Graphs in one line, with no manual buffer management. Manual CUDA Graph capture is an optimisation to reach for only when:

1. You've profiled and **confirmed** CPU launch overhead is your bottleneck.
2. `reduce-overhead` is not available (e.g. dynamic shapes the compiler can't prove are static).
3. You need to capture a scope the compiler doesn't see (multiple forward passes, a full serving loop).

But the notebook lets you **measure** instead of guess — run it and see for yourself on your own GPU.